# 02 Model Comparison

탐색 단계에서 확인한 최신 산출물을 기반으로 후보 모델을 비교합니다. 행동 인식은 랜드마크 기반 MLP/TCN을 비교하고, YOLO는 가벼운 백본 후보를 비교합니다.

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/sessac_project')
ARTIFACT_DIR = Path('/content/drive/MyDrive/sessac_project_artifacts/model_comparison')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR / '3_models') not in sys.path:
    sys.path.append(str(PROJECT_DIR / '3_models'))

from colab_paths import resolve_project_paths
from behavior_modeling import (
    build_dataloaders,
    build_group_folds,
    discover_behavior_samples,
    fit_model,
    load_behavior_arrays,
    summarize_history,
)
from model_registry import get_current_behavior_model_builders, get_legacy_behavior_model_specs
from yolo_workflow import prepare_yolo_dataset, train_yolo_model, write_yolo_yaml

paths = resolve_project_paths(PROJECT_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
records = discover_behavior_samples(
    labels_root=paths['behavior_labels'],
    landmarks_root=paths['behavior_landmarks'],
    legacy_labels_root=paths['legacy_labels'],
    legacy_landmarks_root=paths['legacy_landmarks'],
)
data_dict = load_behavior_arrays(records)
folds = build_group_folds(records, num_folds=4)

current_model_builders = get_current_behavior_model_builders()
legacy_model_specs = get_legacy_behavior_model_specs()

catalog_rows = [
    {
        'model': key,
        'source': 'current',
        'family': key,
        'legacy_mean_val_acc': None,
        'notes': 'current comparison candidate',
    }
    for key in current_model_builders
]
catalog_rows.extend(
    {
        'model': spec.key,
        'source': 'legacy',
        'family': spec.family,
        'legacy_mean_val_acc': spec.mean_val_acc,
        'notes': spec.notes,
    }
    for spec in legacy_model_specs
)
catalog_df = pd.DataFrame(catalog_rows)
catalog_df.to_csv(ARTIFACT_DIR / 'behavior_model_catalog.csv', index=False, encoding='utf-8-sig')
display(catalog_df)

model_builders = dict(current_model_builders)
model_builders.update({spec.key: spec.builder for spec in legacy_model_specs})

comparison_rows = []
for model_name, model_builder in model_builders.items():
    for fold_info in folds:
        train_dataset, val_dataset, train_loader, val_loader = build_dataloaders(
            data_dict=data_dict,
            fold_info=fold_info,
            batch_size=64,
            window_size=15,
            step_size=5,
        )
        sample_batch = next(iter(train_loader))
        input_dim = sample_batch['x'].shape[-1]
        num_classes = sample_batch['y_last'].shape[-1]
        model = model_builder(input_dim, num_classes).to(device)
        model, history = fit_model(model, train_loader, val_loader, device, epochs=50, patience=8)
        summary = summarize_history(history)
        comparison_rows.append({
            'model': model_name,
            'source': 'legacy' if model_name.startswith('legacy_') else 'current',
            'fold': fold_info['fold_index'],
            **summary,
            'train_windows': len(train_dataset),
            'val_windows': len(val_dataset),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(ARTIFACT_DIR / 'behavior_model_comparison.csv', index=False, encoding='utf-8-sig')
display(comparison_df)
display(comparison_df.groupby(['source', 'model'])[['best_val_loss', 'best_val_acc']].mean().sort_values('best_val_loss'))

In [ ]:
yolo_compare_root = ARTIFACT_DIR / 'yolo_compare'
dataset_info = prepare_yolo_dataset(
    image_root=paths['yolo_images'],
    label_root=paths['yolo_labels_open_close'],
    output_root=yolo_compare_root / 'dataset_open_close',
    val_ratio=0.2,
    seed=42,
)
yaml_path = write_yolo_yaml(
    dataset_root=dataset_info['dataset_root'],
    yaml_path=yolo_compare_root / 'open_close.yaml',
    class_names=['open', 'close'],
)

yolo_rows = []
for model_name in ['yolov8n.pt', 'yolov8s.pt']:
    result = train_yolo_model(
        yaml_path=yaml_path,
        model_name=model_name,
        project_dir=yolo_compare_root / 'runs',
        run_name=Path(model_name).stem,
        epochs=20,
        image_size=640,
        batch_size=16,
        device=0,
    )
    yolo_rows.append({'model': model_name, **result})

yolo_df = pd.DataFrame(yolo_rows)
yolo_df.to_csv(ARTIFACT_DIR / 'yolo_model_comparison.csv', index=False, encoding='utf-8-sig')
display(yolo_df)